Goal: replicate interpolation performed in iMOD5 with imod_python.

Reason: I noticed that they're a bit different (3% in g:\models\NBr\PoP\Out\NBr66\WB\WB_Diff_NBr66_m_NBr59_19961231_cumulative.xlsx).

# 0. Init

In [24]:
import numpy as np
import pandas as pd
import primod
from imod import mf6, msw
from WS_Mdl.core.mdl import Mdl_N
from WS_Mdl.core.runtime import timed_Exe
from WS_Mdl.core.style import sprint
from WS_Mdl.imod.mf6.solution import moderate_settings
from WS_Mdl.imod.msw.mete_grid import Cvt_to_AbsPa, add_missing_Cols
from WS_Mdl.imod.prj import o_with_OBS, regrid, regrid_DA

In [10]:
from WS_Mdl.imod.mf6.headers import d_Pkg_Cols
from WS_Mdl.imod.xr import clip_Mdl_area
from WS_Mdl.imod.ini import CeCes
from importlib import reload as RL
import matplotlib.pyplot as plt
import WS_Mdl.core.df

In [3]:
MdlN = 'NBr67'
M = Mdl_N(MdlN)
M.verbose = True
M.Bin_Ins = False

In [5]:
MB = Mdl_N('NBr59')

In [7]:
x_CeCes, y_CeCes = CeCes(MdlN)

🟢 - INI file G:\models\NBr\code\Mdl_Prep\Mdl_Prep_NBr67.ini read successfully. Dictionary created with 21 keys.


# 1. Load PRJ

In [4]:
PRJ_ = timed_Exe(
    o_with_OBS,
    M.Pa.PRJ,
    pre=f'  - Loading {M.Pa.PRJ.name} ...',
    verbose_in=True,
    verbose_out=M.verbose,
    post='🟢',
)
PRJ, period_data = PRJ_[0], PRJ_[1]

PRJ_regrid = timed_Exe(
    regrid,
    PRJ,
    M.MdlN,
    pre='  - Regridding PRJ ...',
    post='',
    verbose_in=True,
    verbose_out=M.verbose,
)  # Speeds up Mdl load.

# Set outer boundaries to -1. Otherwise CHD won't be loaded properly.
BND = PRJ_regrid['bnd']['ibound']
BND.loc[:, [BND.y[0], BND.y[-1]], :] = -1  # Top and bottom rows
BND.loc[:, :, [BND.x[0], BND.x[-1]]] = -1  # Left and right columns
# sprint('🟢', verbose_in=True, verbose_out=M.verbose, print_time=True)

  - Loading NBr67.prj ... 🟢 [150.7s]
  - Regridding PRJ ...  [0.7s]


# Load CHDs from iMOD5

In [55]:
d_CHD_5 = {i.parent.name: pd.read_csv(i,
                               names=[col[0] for col in d_Pkg_Cols['CHD']]+['aux1', 'aux2'],
                               sep=r'\s+',
                               skipfooter=12,  # Footer contains dimensions
                               engine='python') for i in list((MB.Pa.Sim_In / 'CHD6').rglob('*CHD_T14.ARR'))}
DF_CHD_5 = pd.concat([i for i in d_CHD_5.values() if not i.empty], ignore_index=True).drop(columns=['aux1', 'aux2'])
A_CHD_XY = DF_CHD_5.ws.Calc_XY(M.Xmin, M.Ymax, M.cellsize).drop(columns=['i', 'j']).set_index(['k', 'y','x']).to_xarray().sel(k=1)

# Load CHDs from imod python

In [56]:
DA = PRJ['chd-1']['head'] # Load CHD to DA for easier access
DA = DA.sel(x=slice(M.Xmin-abs(DA.dx), M.Xmax+abs(DA.dx)), y=slice(M.Ymax+abs(DA.dy), M.Ymin-abs(DA.dy))) # Clip to Mdl area + 1 cell bufer for interpolation.

# Comapare

In [58]:
DF = pd.DataFrame()
for method in ['linear', 'slinear', 'cubic', 'quintic']:
    Diff = (regrid_DA(DA, x_CeCes, y_CeCes, M.cellsize, M.cellsize, 'chd1.head', method=method).sel(time='1996-01-14') - A_CHD_XY['head'])
    DF[method] = {i: np.nanpercentile(Diff, i) for i in [0, 5, 10, 50, 90, 95, 100]}

In [59]:
Diff_Y_then_X = DA.interp(y=y_CeCes, method='linear').interp(x=x_CeCes, method='linear').sel(time='1996-01-14') - A_CHD_XY['head']
Diff_X_then_Y = DA.interp(x=x_CeCes, method='linear').interp(y=y_CeCes, method='linear').sel(time='1996-01-14') - A_CHD_XY['head']
DF['Y_then_X'] = {i: np.nanpercentile(Diff_Y_then_X, i) for i in [0, 5, 10, 50, 90, 95, 100]}
DF['X_then_Y'] = {i: np.nanpercentile(Diff_X_then_Y, i) for i in [0, 5, 10, 50, 90, 95, 100]}

In [60]:
DF

,linear,slinear,cubic,quintic,Y_then_X,X_then_Y
0,-0.156432,-0.156432,-0.253905,-0.317610,-0.156432,-0.156432
5,-0.114453,-0.114453,-0.131760,-0.183666,-0.114453,-0.114453
10,-0.069683,-0.069683,-0.101777,-0.129280,-0.069683,-0.069683
50,0.000085,0.000085,-0.000337,0.000384,0.000085,0.000085
90,0.030795,0.030795,0.069069,0.113026,0.030795,0.030795
95,0.039179,0.039179,0.099789,0.149849,0.039179,0.039179
100,0.070610,0.070610,0.179361,0.375197,0.070610,0.070610
